# Các phương pháp khắc phục sự cố khi bị tấn công trên model Logistic Regression

Sau khi đã "nhận biết" (awareness) được rằng dữ liệu huấn luyện có thể bị đầu độc bằng label flipping (xem `anwareness-attack.ipynb`), notebook này thử nghiệm hai hướng **phòng thủ (defend)** khác nhau, đều nhằm mục tiêu khôi phục lại accuracy gần với mức baseline sạch (~96%), ngay cả khi vẫn phải huấn luyện trên dữ liệu đã bị đầu độc ở mức nặng — **45% nhãn bị đảo** (`train_dfs["45"]`), mức cao nhất trong các thí nghiệm trước.
 
Cả hai kỹ thuật đều tự fit riêng một `StandardScaler` trên `X_train` của đúng mức % poisoned đang xét, rồi áp dụng scaler đó lên `X_test` để tạo `X_test_scaled` — đảm bảo mô hình huấn luyện và đánh giá luôn nằm trên **cùng một thang đo**, tránh sai lệch kết quả do lệch scale giữa train/test.

In [ ]:
import sys, os 
from pathlib import Path

project_root = Path.cwd().parent 

os.chdir(project_root)
sys.path.append(str(project_root))

In [ ]:
# Python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Project
from src.defend import *
from src.preprocess import X_test, y_test, train_dfs
from src.utils.math import *
from src.utils.config import config

## 1. Regularization Sweep

**Ý tưởng:** khi mô hình bị buộc học từ nhãn nhiễu, nó có xu hướng đẩy trọng số lên cao để cố "giải thích" cho cả các điểm mâu thuẫn (đã quan sát ở Case 2 của `anwareness-attack.ipynb`: `||w||` tăng mạnh theo % flip). Regularization (phạt độ lớn trọng số) là công cụ kinh điển để kìm hãm xu hướng này. Case này quét qua nhiều giá trị `C` (tham số nghịch đảo của độ mạnh regularization trong `sklearn` — `C` càng nhỏ, regularization càng mạnh) để xem mức regularization nào giúp mô hình "kháng cự" tốt nhất trước dữ liệu bị đầu độc 45%.

### 1.1. Chuẩn bị

Hàm `rs(poisoned_level, values)`:
- Lấy `X_train`, `y_train` từ `train_dfs[str(poisoned_level)]`, scale bằng `StandardScaler` fit riêng trên chính tập này.
- Với mỗi giá trị `C` trong `values` (lấy từ `config["C-values"]`), gọi `regularization_sweep(X_train, y_train, X_test_scaled, y_test, C)` — hàm này (định nghĩa trong `src/defend`) huấn luyện một Logistic Regression với hệ số `C` tương ứng, trả về `accuracy`, chuẩn trọng số `w_norm` trên tập test.
- Chạy với `poisoned_level = 45`.

In [ ]:
def rs(poisoned_level, values):
    result = []
    
    dt_train = train_dfs[str(poisoned_level)]

    sc = StandardScaler()

    X_train, y_train = dt_train["X"], dt_train["attack"]
    
    X_train = sc.fit_transform(X_train)
    X_test_scaled = sc.transform(X_test)

    for val in values:
        acc, w_norm, _ = regularization_sweep(
            X_train, 
            y_train,
            X_test_scaled,
            y_test,
            val
        )

        result.append({
            "val": val,
            "acc": acc,
            "w_norm": w_norm
        })

    return result

In [ ]:
# Chaing poisoned level to see difference 
pl_rs = 45                        
C_vals = config["C-values"]

results_C = rs(pl_rs, C_vals)
results_C_df = pd.DataFrame(results_C)

### 1.2. Vẽ biểu đồ

**Kết quả:**
 
| C | Accuracy |
|---|---|
| 10⁻³ (regularization rất mạnh) | **≈ 0.630** |
| 10⁻² | ≈ 0.597 |
| 10⁻¹ | ≈ 0.579 |
| 10⁰ – 10⁴ (regularization yếu → gần như không có) | dao động ổn định quanh **≈ 0.575 – 0.583** |

Xu hướng rõ ràng: **`C` càng nhỏ (regularization càng mạnh) → accuracy càng cao**. Ở `C = 10⁻³`, accuracy đạt đỉnh ~63% — cao hơn khoảng 5 điểm % so với mức baseline không regularization đáng kể (~58%, tương ứng vùng `C ≥ 1`). Khi `C` tăng dần (regularization yếu đi), accuracy giảm và ổn định ở một mặt bằng thấp hơn, phản ánh đúng giả thuyết ban đầu: phạt độ lớn trọng số giúp mô hình không "cố" khớp quá mức với các điểm nhãn bị đảo, từ đó giữ được phần nào khả năng tổng quát hóa trên dữ liệu sạch.
 
Mức cải thiện ~5 điểm % là **có thật nhưng khiêm tốn** — regularization một mình không thể đưa accuracy quay lại gần mức 96% của mô hình sạch, cho thấy đây là một biện pháp giảm thiểu (mitigation) một phần, không phải giải pháp triệt để cho tấn công label flipping ở mức 45%.

In [ ]:
plt.plot(results_C_df["val"], results_C_df["acc"], marker="o", label="Accuracy")
plt.xscale("log")   
plt.xlabel("C (small = strong regularization)")
plt.ylabel("Accuracy")
plt.title(f"Effect level of Regularization onto poisoned {pl_rs}%")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 2. Self - Cleaning

**Ý tưởng:** thay vì kìm hãm mô hình bằng regularization, hướng tiếp cận này cố **lọc bỏ trực tiếp** những mẫu huấn luyện bị nghi ngờ là nhãn sai, trước khi huấn luyện — dựa trên một tiêu chí nào đó (hàm `cleaning()` trong `src/defend`, ví dụ có thể dựa trên độ tin cậy dự đoán hoặc mức độ bất thường của mẫu). Tham số `keep_ratio` quyết định giữ lại bao nhiêu % dữ liệu train sau khi lọc (giữ lại các mẫu "đáng tin cậy" nhất theo tiêu chí đó).

### 2.1. Chuẩn bị

Hàm `sc(poisoned_level, krs)`:
- Lấy `X_train`, `y_train` từ `train_dfs[str(poisoned_level)]`, scale bằng `StandardScaler` riêng.
- Huấn luyện một mô hình **baseline** (`lr_p`, không lọc gì cả) trên toàn bộ `X_train` đã scale, tính `acc_b` trên `X_test_scaled` — dùng làm đường tham chiếu "trước khi lọc".
- Với mỗi `keep_ratio` trong `krs` (lấy từ `config["keep-ratios"]`), gọi `cleaning(X_train, y_train, keep_ratio)` để lọc ra `X_cleaned`, `y_cleaned`, rồi huấn luyện lại Logistic Regression trên tập đã lọc và đánh giá trên `X_test_scaled`.
- Có bước bảo vệ: nếu sau khi lọc mà `y_cleaned` chỉ còn 1 lớp duy nhất (lọc quá tay, mất hết một nhãn) thì bỏ qua điểm đó, tránh lỗi khi huấn luyện.
- Chạy với `poisoned_level = 45`.

In [ ]:
def sc(poisoned_level, krs):
    result = []
    
    dt_train = train_dfs[str(poisoned_level)]

    X_train, y_train = dt_train["X"], dt_train["attack"]

    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test_scaled = sc.transform(X_test)

    lr_p = LogisticRegression().fit(X_train, y_train)
    acc_b = accuracy_score(y_test, lr_p.predict(X_test_scaled))

    for kr in krs:
        X_cleaned, y_cleaned = cleaning(
            X_train, 
            y_train,
            kr
        )

        # If cleaning only have one style,
        # This mean just has 0 or 1
        # We will skip this data
        if len(np.unique(y_cleaned)) < 2: continue

        lr_cleaned = LogisticRegression().fit(X_cleaned, y_cleaned)

        result.append({
            "val": kr,
            "acc": accuracy_score(y_test, lr_cleaned.predict(X_test_scaled)),
            "w_norm": np.linalg.norm(lr_cleaned.coef_)
        })

    return acc_b, result

In [ ]:
# Chaing poisoned level to see difference 
pl_sc = 45    
keep_ratios = config["keep-ratios"]

acc_b, result_kr = sc(pl_sc, keep_ratios)
results_kr_df = pd.DataFrame(result_kr)

### 2.2. Vẽ biểu đồ

**Kết quả:** (baseline trước khi lọc: **acc_b ≈ 0.582**)
 
| % Keep | Accuracy | So với baseline |
|---|---|---|
| 0.65 | ≈ 0.603 | cao hơn |
| 0.70 | ≈ 0.566 | thấp hơn |
| 0.75 | ≈ 0.591 | cao hơn |
| 0.80 | ≈ 0.578 | thấp hơn (nhẹ) |
| 0.85 | ≈ 0.570 | thấp hơn |
| 0.90 | ≈ 0.590 | cao hơn |
| 0.95 | ≈ 0.576 | thấp hơn (nhẹ) |
 
Không giống Regularization Sweep (có xu hướng đơn điệu, dễ diễn giải), kết quả Self-Cleaning ở đây **dao động quanh baseline** mà không cho thấy một xu hướng rõ ràng theo `keep_ratio` — có điểm cải thiện nhẹ (0.65, 0.75, 0.90), có điểm tệ hơn cả baseline (0.70, 0.85). Biên độ dao động (0.566 – 0.603, tức khoảng ±2 điểm % quanh baseline) khá nhỏ so với mức cải thiện ~5 điểm % quan sát được ở Regularization Sweep.
 
Điều này gợi ý vài khả năng cần kiểm chứng thêm trước khi kết luận Self-Cleaning "có tác dụng" hay không:
- Kết quả mới chạy **1 lần duy nhất** cho mỗi `keep_ratio` — nên lẫn nhiễu ngẫu nhiên từ chính cách hàm `cleaning()` chọn mẫu để loại bỏ ở từng ngưỡng. Nên chạy lặp lại nhiều lần (hoặc trên nhiều tập con) rồi lấy trung bình ± độ lệch chuẩn để có kết luận đáng tin cậy hơn.
- Có thể tiêu chí lọc trong `cleaning()` chưa đủ tốt để phân biệt mẫu bị đảo nhãn khỏi mẫu hợp lệ ở mức đầu độc 45% (tỉ lệ nhiễu quá cao khiến tiêu chí lọc mất độ chính xác).

In [ ]:
plt.plot(results_kr_df["val"], results_kr_df["acc"], marker="o")
plt.axhline(acc_b, color="red", linestyle="--", label=f"Before filtering ({acc_b:.2f})")
plt.xlabel("% Keep")
plt.ylabel("Accuracy")
plt.title(f"Effective after filtering poisoned label at {pl_sc}%")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Kết luận

So sánh hai hướng phòng thủ trên cùng mức đầu độc 45%:
 
- **Regularization Sweep** cho kết quả **nhất quán và dễ diễn giải**: giảm `C` (tăng regularization) cải thiện accuracy rõ rệt và ổn định, từ ~58% lên đến ~63% ở mức regularization mạnh nhất. Đây là biện pháp đơn giản, rẻ (chỉ cần đổi 1 tham số khi huấn luyện), và cho hiệu quả có thể dự đoán được.
- **Self-Cleaning** có tiềm năng cải thiện cao hơn về mặt lý thuyết (loại bỏ trực tiếp mẫu nhiễu thay vì chỉ kìm hãm mô hình), nhưng kết quả thực nghiệm hiện tại **chưa đủ ổn định** để khẳng định hiệu quả — dao động quanh baseline nhiều hơn là cải thiện rõ rệt, cần thêm thực nghiệm (lặp lại nhiều lần, thử các tiêu chí lọc khác nhau) để đánh giá đúng tiềm năng của hướng này.
Cả hai kỹ thuật đều **không đưa accuracy quay lại gần mức baseline sạch (~96%)** — cho thấy ở mức đầu độc nặng (45%), các biện pháp phòng thủ đơn lẻ (regularization hoặc lọc dữ liệu) chỉ giảm thiểu được một phần thiệt hại, không thể vô hiệu hóa hoàn toàn tấn công label flipping. Hướng phát triển tiếp theo hợp lý là **kết hợp cả hai** (lọc dữ liệu trước, rồi huấn luyện với regularization phù hợp trên phần dữ liệu đã lọc) để xem hiệu quả cộng hưởng có vượt trội hơn từng biện pháp riêng lẻ hay không.